# Model_Akaike (May 8, 2022)

Code by Caroline Juang (c.juang@columbia.edu)

Creating a model that will perform a stepwise regression, evaluating each step using AIC (Akaike information criterion).

This model will use **climate to predict burned area**

In [1]:
# import
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
from scipy.stats import linregress
import joblib # for saving models

## User input

**Make sure to update `climindname` to the correct climate index**

In [2]:
# customize seasons for climate variables

# customize number of rolling periods
ant_years = 1 # antecedent years to include (2 antecedent years + current year)
ant_season = 3 # n+1 of months to include in each period (e.g. input 2 would mean 3 months)

finalyear = 2020 # last year of good WUMI data
modisyear = 2022 # year to predict using MODIS data

# importing data string
directory = 'your_data_folder' # customize here
# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindname = f.read().strip()
print(climindname +' will be used for the SST gradient')
data_string = 'data//'
model_string = 'model//'+climindname+'//'

patch125-155_nino3-34 will be used for the SST gradient


## Scripts

In [3]:
# average climate variables, within a selected season

def annual_seasonAvg(data, firstmonth, finalmonth):
    """
    This intakes an array of current ecoregion's climate variable of monthly averages (data), 
    Output: an array of yearly averages of the ecoregion climate variable, in the
    seasons specified (firstmonth, finalmonth).
    Requirements: time = an xarray timeseries of months in datetime format.
    """
    withinyear = (time['time.year']>= years[0]) & (time['time.year'] <= years[-1])
    withinseason = (time['time.month'] >= firstmonth) & (time['time.month'] <= finalmonth)
    thistime = time[withinseason & withinyear] # cut time
    
    # create pd dataframe based on data, for time resampling
    thisdf = pd.DataFrame({'time': pd.to_datetime(thistime.values), 'clim':data[withinyear & withinseason]}).set_index('time')
    thisdf = thisdf.resample('Y').mean().reset_index()
    return thisdf.clim.values

In [4]:
# average climate variables, within a selected season

def annual_seasonSum(data, firstmonth, finalmonth):
    """
    This intakes an array of current ecoregion's climate variable of monthly averages (data), 
    Output: an array of yearly averages of the ecoregion climate variable, in the
    seasons specified (firstmonth, finalmonth).
    Requirements: time = an xarray timeseries of months in datetime format.
    """
    withinyear = (time['time.year']>= years[0]) & (time['time.year'] <= years[-1])
    withinseason = (time['time.month'] >= firstmonth) & (time['time.month'] <= finalmonth)
    thistime = time[withinseason & withinyear] # cut time
    
    # create pd dataframe based on data, for time resampling
    thisdf = pd.DataFrame({'time': pd.to_datetime(thistime.values), 'clim':data[withinyear & withinseason]}).set_index('time')
    thisdf = thisdf.resample('Y').sum().reset_index()
    return thisdf.clim.values

In [5]:
# calculate AICc for variables

def getAICc(rvalue, nyears, npredictors):
    """
    Calculate the AICc, which is the Akaike Information Criterion (AIC)
    with bias correction for small sample sizes (AICc). 
    Inputs are the variables needed for the calculation. 
    Output is a single value of the AICc.
    r = r-value calculated from a Pearson Correlation
    nyears = number of years in the timeseries
    npredictors = number of x-variables used in the model
    """
    # calculate AIC
    aic = nyears * np.log(1-rvalue**2) + 2*(npredictors+1)
    # calculate AIC with bias correction for small sample size
    aicc = aic + (2*(npredictors+1)*(npredictors+2)) / (nyears - npredictors - 2)
    return aicc

In [6]:
# average climate variables, within a selected season

def annualDetrend(data):
    """
    This intakes an array of current ecoregion's climate variable of annual averages (data), 
    Output: an array of the yearly data, detrended so the slope of the data is zero.
    Requirements: 
    """

    # detrend the data
    xnum = np.arange(0,len(data))
    reg = linregress(xnum, data)
    m = reg.slope
    b = reg.intercept
    regpredict = m*xnum + b
    # normalize by subtracting the linear regression
    tmpdatanorm = (data - regpredict) + b

    return tmpdatanorm

In [7]:
# get amount in forest

def in_allregion(variable, thisregion):
    """
    For getting the average values only in this particular region of the western US.
    Requirements: the western US study area.
    Returns one weighted average of the variable for each timestep.
    """
    thisregion = westUS*thisregion # get in the study area
    weightedsum = (variable*thisregion).sum(dim=['X','Y'], skipna=True)
    regionsum = thisregion.sum()
    return  weightedsum / regionsum

def in_forest(variable, thisregion):
    """
    For getting the average values only in a forest area.
    Requirements: the forest netCDF.
    Returns one weighted average of the variable for each timestep.
    """
    forestregion = forest*thisregion # get in the region
    weightedsum = (variable*(forestregion)).sum(dim=['X','Y'], skipna=True)
    forestsum = (forestregion).sum()
    return  weightedsum / forestsum


def in_nonforest(variable, thisregion):
    """
    For getting the average values only in a nonforested area in the western US.
    Requirements: the forest netCDF, a mask of the western US.
    Returns one weighted average of the variable for each timestep and the selected region.
    """
    nonforestregion = nonforest*thisregion # get in the region
    weightedsum = (variable*(nonforestregion)).sum(dim=['X','Y'], skipna=True)
    nonforestsum = (nonforestregion).sum()
    return  weightedsum / nonforestsum

## import ecoregion data

In [8]:
# import western US study area
westUS_string = '12km//study_area//westUS.nc'
westUS = xr.open_dataset(directory + westUS_string, engine='netcdf4')
westUS = westUS.westUS

# import forest
forest_string = '12km//landcover//US_ForestType_Ruefenacht//forest_type_frac.nc'
forest = xr.open_dataset(directory + forest_string, engine='netcdf4')
forest = forest.forest_type_frac.sum(dim='ftype') # sum over all forest types

# create forest and nonforest area 
forest = westUS*forest
nonforest = westUS*(1-forest)

In [9]:
# create strings of the types of forests, remove #5
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']
province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

## Import fire, forest data
from `Data_CreateModelData`

In [10]:
# import fire data, both modis and wumi databases
filename = data_string + 'WUMI-ecoprovinces'
wumi = pd.read_csv(filename+'_all.txt').set_index('Unnamed: 0')
wumifor = pd.read_csv(filename+'_for.txt').set_index('Unnamed: 0')
wuminon = pd.read_csv(filename+'_non.txt').set_index('Unnamed: 0')

## Import climate data
from `Data_CreateModelData`

Exclude specific variables:
(7/16/2024) Exclude Tmax, Tmin, RH and all prior-year variables except prec

In [11]:
# import model data

dfframesall = {}
dfframesfor = {}
dfframesnon = {}


for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climate_ecoprovinces_'
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')


# in km2, the minimum fire sum for model to be valid
minfiresum = 200

# variables with known correlation with burned area in forest 
# in current year
clim_negcorr = ['wetdays', 'prec', 'rh'] # should be negative
clim_poscorr = ['vpd', 'tmax', 'tmin', 'solar'] # should be positive
clim_poscorrboth = ['wind'] # forest and nonforest
clim_correxclude = ['rh y0', 'tmean y0',
                 'rh y-1', 'tmean y-1',
                 'wind y-1','tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1'] # remove most of the prior-year variables

Akaike Information Criterion - https://doi.org/10.1109/TAC.1974.1100705

```AIC = Nyears * log(1-r.^2) + 2*(Npredictors+1)```

* `r^2` = square of the correlation coefficient
* `Nyears` = number of years in the timeseries
* `Npredictors` = the number of timeseries you're using as predictors
* rewarded for large sample size, penalized for adding more number of predictors

regression and time series model selection in small samples: https://doi.org/10.1093/biomet/76.2.297

```AICc = AIC + (2*(Npredictors+1)*(Npredictors+2))/(Nyears - Npredictors - 2)```
* bias correction on the AIC

In [12]:
# storage for each model's variable indices
dfmodelall = {}
dfmodelfor = {}
dfmodelnon = {}

In [13]:
# regress all potential predictors against burned area, for ALL area
print('ALL LANDCOVER')
landname = 'all'
modeloutputfile = model_string + 'modeloutput_burnarea_'+landname+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesall['allwestUS'].columns.values # each climate variable

for iregion in np.arange(len(province_num)):
    print('+++ model for ' + dfnames[iregion] + '\t AICC \t r-value')
    f.write('+++'+dfnames[iregion]+'\n')
    varoutputfile = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.txt' # save variables in model
    f_vars = open(varoutputfile, 'w') # print to variable data file
    tmpr = np.zeros(len(labels)) # storage for regression correlation
    tmpr_clim = np.zeros(len(labels)) # storage for r corr, clim vs. BA
    tmpr_climDT = np.zeros(len(labels)) # storage for r corr, climDT vs. BA_DT
    aicc = np.zeros(len(labels)) # storage for aicc values for each set of reg fit
    tmpaicc = [] # storage for aicc values for each model
    tmpi = [] # empty list of indices for variables

    # get observed ecoregion burned area, but set min value
    tmpfire = np.asarray(np.ma.masked_invalid(np.log10(wumi.iloc[:,iregion])).filled(0))
    tmpfiresum = wumi.iloc[:,iregion].sum()
    tmpfiremin = tmpfire[tmpfire!=0].min()
    tmpfire[tmpfire==0] = tmpfiremin # replace 0 with minimum nonzero value
    tmpfireDT = annualDetrend(tmpfire)
    
    for i, label in enumerate(labels):
        tmpclim = np.asarray(dfframesall[dfnames[iregion]][label].values.reshape(-1,1))
        tmpclimDT = annualDetrend(tmpclim.flatten())
        reg = LinearRegression() # least squares
        reg.fit(tmpclim, tmpfire)
        tmppredict = reg.predict(tmpclim)
        tmpr[i] = pearsonr(tmpfire.flatten(), tmppredict.flatten()).statistic
        # calculate AICc for each
        nyears = len(tmpfire)
        npredictors = reg.n_features_in_
        aicc[i] = getAICc(tmpr[i], nyears, npredictors)
        # compare signs
        tmpr_clim = pearsonr(tmpclim.flatten(), tmpfire)
        tmpr_climDT = pearsonr(tmpclimDT, tmpfireDT)
        
        # check if the correlation is right sign intuitively
        tmplabeljustvar = label.split(' ')[0]
        tmplabeljustyr = label.split(' ')[1]
        tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
        # wrong signs
        tmpwrong1 = (tmpr_clim.statistic > 0) & (tmplabeljustvar in clim_negcorr)
        tmpwrong2 = (tmpr_clim.statistic < 0) & (tmplabeljustvar in clim_poscorr)
        tmpwrong3 = (tmpr_clim.statistic < 0) & (tmplabeljustvar in clim_poscorrboth)
        tmpwrong4 = (tmpnameyrlabel in clim_correxclude)
        if tmpwrong1 or tmpwrong2 or tmpwrong3 or tmpwrong4:
            aicc[i] = 99 # overwrite the AICc value
            
        # check if they are the same sign
        tmpsign = ((tmpr_clim.statistic > 0) == (tmpr_climDT.statistic > 0))
        tmppvalue = (tmpr_climDT.pvalue < 0.1) # check significance
        #if tmpsign==False:
        #    print('\tSIGN CHANGE ' + label + '\t'+
        #              'r={:.5f} \t r={:.5f}'.format(tmpr_clim.statistic, tmpr_climDT.statistic))
        #if tmppvalue==False:
        #    print('\tDT INSIGNIFICANT ' + label + '\t'+
        #              'p={:.5f} \t p={:.5f}'.format(tmpr_clim.pvalue, tmpr_climDT.pvalue))
        if (tmpsign==False) or (tmppvalue==False):
            aicc[i] = 99 # overwrite the AICc value
            #print('\tOverwrite AICc')
    # get minimum
    [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
    tmpi.append(iminaicc) # add min to list of variables
    tmpaicc.append(minaicc) # add aicc value to list
    f.write(labels[iminaicc] + '\n')
    

    if (minaicc>0) or (tmpfiresum<minfiresum): # we overwrite the model and EXIT!!!
        # need all write/output requirements because we're EXITING!!!
        tmpclim = np.asarray(dfframesall[dfnames[iregion]].iloc[:,tmpi].values.reshape(-1,1))
        reg = LinearRegression()
        reg.fit(tmpclim, tmpfire)
        reg.coef_ = np.asarray([0]) # change coeff to 0
        reg.intercept_ = np.asarray(tmpfire.mean())
        f_vars.write(labels[iminaicc] + '\n' + str(tmpclim.flatten()) + '\n') # write variable data to file
        print('\tAICc POSITIVE--EXIT')
        print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
              str(np.round(tmpr[iminaicc], decimals=3)))
        
        # store the final model
        filename = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.sav' # save model
        joblib.dump(reg, filename) # save model        

        tmpcoef = reg.coef_
        tmpintercept = reg.intercept_
        tmppred = reg.predict(tmpclim)
        print('R-value ' + str(pearsonr(tmpfire.flatten(), tmppred.flatten())[0]))
        # write to file
        f_vars.write('Model parameters \n' + str(tmpcoef) + '\n')
        f_vars.write('Model intercept\n' + str(tmpintercept) + '\n')
        f_vars.write('Burned area \n' + str(tmpfire) + '\n')
        f_vars.write('Model Burned area prediction \n' + str(tmppred))
        dfmodelnon[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list0.
        f_vars.close() # close variable data file
        continue # leave this loop
    
    # calculate residuals from one-variable model
    tmpclim = dfframesall[dfnames[iregion]].iloc[:,tmpi]
    reg = LinearRegression()
    f_vars.write(labels[iminaicc] + '\n' + str(tmpclim.values.flatten()) + '\n') # write variable data to file
    reg.fit(tmpclim, tmpfire)
    tmppredictprior = reg.predict(tmpclim)

    print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
              str(np.round(tmpr[iminaicc], decimals=3)))
    
    # define starting difference in aicc to get the for-loop to start
    diffaicc = -3
    # multiple regression using remaining predictor, until aicc is not improved
    while diffaicc < -2:
        # add another variable to get new model
        # calculate residuals
        tmpresids = np.log10((10**tmpfire)/(10**tmppredictprior))
        # go through each variable and get the correlation
        tmpr = np.zeros(len(labels)) # storage for regression correlation
        for i, label in enumerate(labels):
            tmpclimnew = np.asarray(dfframesall[dfnames[iregion]][label]).reshape(-1,1)
            tmpclimprior = dfframesall[dfnames[iregion]].iloc[:,tmpi]
            tmpX = np.hstack((tmpclimprior, tmpclimnew)) # all X variables
            tmpclimnewDT = annualDetrend(tmpclimnew.flatten())
            tmpresidsDT = annualDetrend(tmpresids)
            reg = LinearRegression() # least squares
            reg.fit(tmpX, tmpfire)
            tmppredict = reg.predict(tmpX)
            tmpr[i] = pearsonr(tmpfire.flatten(), tmppredict.flatten()).statistic
            # calculate AICc for each
            nyears = len(tmpfire)
            npredictors = reg.n_features_in_
            aicc[i] = getAICc(tmpr[i], nyears, npredictors)
            # compare signs
            tmpr_clim = pearsonr(tmpclimnew.flatten(), tmpresids)
            tmpr_climDT = pearsonr(tmpclimnewDT, tmpresidsDT)
            # check if the correlation is right sign physically
            tmplabeljustvar = label.split(' ')[0]
            tmplabeljustyr = label.split(' ')[1]
            tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
            # wrong signs
            tmpwrong1 = (tmpr_clim.statistic > 0) & ((tmplabeljustvar in clim_negcorr)&(tmplabeljustyr in ['y0']))
            tmpwrong2 = (tmpr_clim.statistic < 0) & ((tmplabeljustvar in clim_poscorr)&(tmplabeljustyr in ['y0']))
            tmpwrong3 = (tmpr_clim.statistic < 0) & (tmplabeljustvar in clim_poscorrboth)
            tmpwrong4 = (tmpnameyrlabel in clim_correxclude)
            if tmpwrong1 or tmpwrong2 or tmpwrong3 or tmpwrong4:
                aicc[i] = 99 # overwrite the AICc value
            # check if they are the same sign
            tmpsign = ((tmpr_clim.statistic > 0) == (tmpr_climDT.statistic > 0))
            tmppvalue = (tmpr_climDT.pvalue < 0.1) # check significance
            #if tmpsign==False:
            #    print('\tSIGN CHANGE ' + label + '\t'+
            #              'r={:.5f} \t r={:.5f}'.format(tmpr_clim.statistic, tmpr_climDT.statistic))
            #if tmppvalue==False:
            #    print('\tDT INSIGNIFICANT ' + label + '\t'+
            #              'p={:.5f} \t p={:.5f}'.format(tmpr_clim.pvalue, tmpr_climDT.pvalue))
            if (tmpsign==False) or (tmppvalue==False):
                aicc[i] = 99 # overwrite the AICc value
                #print('\tOverwrite AICc')

        # get minimum
        [iminaicc, minaicc] = np.nanargmin(aicc), np.nanmin(aicc)
        # check aicc value to previous value
        diffaicc = minaicc - tmpaicc[-1]
        # check if diff aicc is small enough before continuing
        if (diffaicc >= -2):
            continue # exit loop
        else:
            # continue this part if aicc < -2
            tmpi.append(iminaicc) # add min to list of variables
            tmpaicc.append(minaicc) # add aicc value to list
            f.write(labels[iminaicc] + '\n')
            print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
                      str(np.round(tmpr[iminaicc], decimals=3)))

            # calculate residuals from this current model
            tmpX = dfframesall[dfnames[iregion]].iloc[:,tmpi] # all the variables
            f_vars.write(labels[iminaicc] + '\n' + str(tmpX.iloc[:,-1:].values.flatten()) + '\n') # write var data to file
            reg = LinearRegression()
            reg.fit(tmpX, tmpfire) # fit the final model
            tmppredictprior = reg.predict(tmpX)

    # store the final model
    tmpX = dfframesall[dfnames[iregion]].iloc[:,tmpi] 
    reg.fit(tmpX, tmpfire) # fit the final model
    filename = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.sav' # save model
    joblib.dump(reg, filename) # save model        

    tmploadmodel = joblib.load(filename)
    tmpcoef = tmploadmodel.coef_
    tmpintercept = tmploadmodel.intercept_
    tmppred = tmploadmodel.predict(tmpX)
    print('R-value ' + str(pearsonr(tmpfire.flatten(), tmppred.flatten())[0]))
    # write to file
    f_vars.write('Model parameters \n' + str(tmpcoef) + '\n')
    f_vars.write('Model intercept\n' + str(tmpintercept) + '\n')
    f_vars.write('Burned area \n' + str(tmpfire) + '\n')
    f_vars.write('Model Burned area prediction \n' + str(tmppred))
    dfmodelall[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list0.
    f_vars.close() # close variable data file
f.close()

ALL LANDCOVER
+++ model for allwestUS	 AICC 	 r-value
vpd y0 mo 7-9	-22.915	0.709
vpd y0 mo 4-6	-30.593	0.785
prec y-1 mo 4-6	-33.796	0.817


tmax y0 mo 10-12	-36.823	0.844
prec y-1 mo 7-9	-39.474	0.866
prec y-1 mo 10-12	-43.312	0.889
R-value 0.8889997706412631
+++ model for ecoprov1	 AICC 	 r-value
	AICc POSITIVE--EXIT
vpd y0 mo 7-9	1.13	0.281
R-value nan
+++ model for ecoprov2	 AICC 	 r-value
vpd y0 mo 4-6	-12.117	0.587
tmax y0 mo 7-9	-21.834	0.72
prec y0 mo 10-12	-24.732	0.762


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


R-value 0.7621131484407855
+++ model for ecoprov3	 AICC 	 r-value
	AICc POSITIVE--EXIT
prec y0 mo 4-6	0.492	0.306
R-value nan
+++ model for ecoprov4	 AICC 	 r-value
vpd y0 mo 10-12	-5.735	0.477


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


...

tmin y0 mo 4-6	-37.381	0.835
R-value 0.8348232385074217
+++ model for ecoprov18	 AICC 	 r-value
	AICc POSITIVE--EXIT
tmax y0 mo 10-12	-0.045	0.326
R-value nan
+++ model for ecoprov19	 AICC 	 r-value
vpd y0 mo 7-9	-16.679	0.645
tmax y0 mo 4-6	-30.227	0.782


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


prec y-1 mo 1-3	-37.387	0.835
vpd y0 mo 10-12	-45.937	0.879
R-value 0.8789589415954183
+++ model for ecoprov20	 AICC 	 r-value
vpd y0 mo 7-9	-31.828	0.777
vpd y0 mo 4-6	-39.207	0.832


R-value 0.8316935733934167
+++ model for ecoprov21	 AICC 	 r-value
	AICc POSITIVE--EXIT
wind y0 mo 1-3	0.837	0.293
R-value nan


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [14]:
# same model for FOREST area
print('FOREST')
landname = 'for'
modeloutputfile = model_string + 'modeloutput_burnarea_'+landname+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesfor['allwestUS'].columns.values # each climate variable

for iregion in np.arange(len(province_num)):
    print('+++ model for ' + dfnames[iregion] + '\t AICC \t r-value')
    f.write('+++'+dfnames[iregion]+'\n')
    varoutputfile = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.txt' # save variables in model
    f_vars = open(varoutputfile, 'w') # print to variable data file
    tmpr = np.zeros(len(labels)) # storage for regression correlation
    tmpr_clim = np.zeros(len(labels)) # storage for r corr, clim vs. BA
    tmpr_climDT = np.zeros(len(labels)) # storage for r corr, climDT vs. BA_DT
    aicc = np.zeros(len(labels)) # storage for aicc values for each set of reg fit
    tmpaicc = [] # storage for aicc values for each model
    tmpi = [] # empty list of indices for variables

    # get observed ecoregion burned area, but set min value
    tmpfire = np.asarray(np.ma.masked_invalid(np.log10(wumifor.iloc[:,iregion])).filled(0))
    tmpfiresum = wumifor.iloc[:,iregion].sum()
    tmpfiremin = tmpfire[tmpfire!=0].min()
    tmpfire[tmpfire==0] = tmpfiremin # replace 0 with minimum nonzero value
    tmpfireDT = annualDetrend(tmpfire)
    
    for i, label in enumerate(labels):
        tmpclim = np.asarray(dfframesfor[dfnames[iregion]][label].values.reshape(-1,1))
        tmpclimDT = annualDetrend(tmpclim.flatten())
        reg = LinearRegression() # least squares
        reg.fit(tmpclim, tmpfire)
        tmppredict = reg.predict(tmpclim)
        tmpr[i] = pearsonr(tmpfire.flatten(), tmppredict.flatten()).statistic
        # calculate AICc for each
        nyears = len(tmpfire)
        npredictors = reg.n_features_in_
        aicc[i] = getAICc(tmpr[i], nyears, npredictors)
        # compare signs
        tmpr_clim = pearsonr(tmpclim.flatten(), tmpfire)
        tmpr_climDT = pearsonr(tmpclimDT, tmpfireDT)
        # check if the correlation is right sign intuitively
        tmplabeljustvar = label.split(' ')[0]
        tmplabeljustyr = label.split(' ')[1]
        tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
        # wrong signs
        tmpwrong1 = (tmpr_clim.statistic > 0) & ((tmplabeljustvar in clim_negcorr)&(tmplabeljustyr in ['y0']))
        tmpwrong2 = (tmpr_clim.statistic < 0) & ((tmplabeljustvar in clim_poscorr)&(tmplabeljustyr in ['y0']))
        tmpwrong3 = (tmpr_clim.statistic < 0) & (tmplabeljustvar in clim_poscorrboth)
        tmpwrong4 = (tmpnameyrlabel in clim_correxclude)
        if tmpwrong1 or tmpwrong2 or tmpwrong3 or tmpwrong4:
            aicc[i] = 99 # overwrite the AICc value
        # check if they are the same sign
        tmpsign = ((tmpr_clim.statistic > 0) == (tmpr_climDT.statistic > 0))
        tmppvalue = (tmpr_climDT.pvalue < 0.1) # check significance
        #if tmpsign==False:
        #    print('\tSIGN CHANGE ' + label + '\t'+
        #              'r={:.5f} \t r={:.5f}'.format(tmpr_clim.statistic, tmpr_climDT.statistic))
        #if tmppvalue==False:
        #    print('\tDT INSIGNIFICANT ' + label + '\t'+
        #              'p={:.5f} \t p={:.5f}'.format(tmpr_clim.pvalue, tmpr_climDT.pvalue))
        if (tmpsign==False) or (tmppvalue==False):
            aicc[i] = 99 # overwrite the AICc value
            #print('\tOverwrite AICc')
    # get minimum
    [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
    tmpi.append(iminaicc) # add min to list of variables
    tmpaicc.append(minaicc) # add aicc value to list
    f.write(labels[iminaicc] + '\n')
    
    if (minaicc>0) or (tmpfiresum<minfiresum): # we overwrite the model and exit
        tmpclim = np.asarray(dfframesfor[dfnames[iregion]].iloc[:,tmpi].values.reshape(-1,1))
        reg = LinearRegression()
        reg.fit(tmpclim, tmpfire)
        reg.coef_ = np.asarray([0]) # change coeff to 0
        reg.intercept_ = np.asarray(tmpfire.mean())
        f_vars.write(labels[iminaicc] + '\n' + str(tmpclim.flatten()) + '\n') # write variable data to file
        print('\tAICc POSITIVE--EXIT')
        print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
              str(np.round(tmpr[iminaicc], decimals=3)))
        
        # store the final model
        filename = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.sav' # save model
        joblib.dump(reg, filename) # save model        

        tmpcoef = reg.coef_
        tmpintercept = reg.intercept_
        tmppred = reg.predict(tmpclim)
        print('R-value ' + str(pearsonr(tmpfire.flatten(), tmppred.flatten())[0]))
        # write to file
        f_vars.write('Model parameters \n' + str(tmpcoef) + '\n')
        f_vars.write('Model intercept\n' + str(tmpintercept) + '\n')
        f_vars.write('Burned area \n' + str(tmpfire) + '\n')
        f_vars.write('Model Burned area prediction \n' + str(tmppred))
        dfmodelnon[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list0.
        f_vars.close() # close variable data file
        continue # leave this loop
    
    # calculate residuals from one-variable model
    tmpclim = dfframesfor[dfnames[iregion]].iloc[:,tmpi]
    f_vars.write(labels[iminaicc] + '\n' + str(tmpclim.values.flatten()) + '\n') # write variable data to file
    reg = LinearRegression()
    reg.fit(tmpclim, tmpfire)
    tmppredictprior = reg.predict(tmpclim)

    print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
              str(np.round(tmpr[iminaicc], decimals=3)))
    
    # define starting difference in aicc to get the for-loop to start
    diffaicc = -3
    # multiple regression using remaining predictor, until aicc is not improved
    while diffaicc < -2:
        # add another variable to get new model
        # calculate residuals
        tmpresids = np.log10((10**tmpfire)/(10**tmppredictprior))
        # go through each variable and get the correlation
        tmpr = np.zeros(len(labels)) # storage for regression correlation
        for i, label in enumerate(labels):
            tmpclimnew = np.asarray(dfframesfor[dfnames[iregion]][label]).reshape(-1,1)
            tmpclimprior = dfframesfor[dfnames[iregion]].iloc[:,tmpi]
            tmpX = np.hstack((tmpclimprior, tmpclimnew)) # all X variables
            tmpclimnewDT = annualDetrend(tmpclimnew.flatten())
            tmpresidsDT = annualDetrend(tmpresids)
            reg = LinearRegression() # least squares
            reg.fit(tmpX, tmpfire)
            tmppredict = reg.predict(tmpX)
            tmpr[i] = pearsonr(tmpfire.flatten(), tmppredict.flatten()).statistic
            # calculate AICc for each
            nyears = len(tmpfire)
            npredictors = reg.n_features_in_
            aicc[i] = getAICc(tmpr[i], nyears, npredictors)
            # compare signs
            tmpr_clim = pearsonr(tmpclimnew.flatten(), tmpresids)
            tmpr_climDT = pearsonr(tmpclimnewDT, tmpresidsDT)
            # check if the correlation is right sign intuitively
            tmplabeljustvar = label.split(' ')[0]
            tmplabeljustyr = label.split(' ')[1]
            tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
            # wrong signs
            tmpwrong1 = (tmpr_clim.statistic > 0) & ((tmplabeljustvar in clim_negcorr)&(tmplabeljustyr in ['y0']))
            tmpwrong2 = (tmpr_clim.statistic < 0) & ((tmplabeljustvar in clim_poscorr)&(tmplabeljustyr in ['y0']))
            tmpwrong3 = (tmpr_clim.statistic < 0) & (tmplabeljustvar in clim_poscorrboth)
            tmpwrong4 = (tmpnameyrlabel in clim_correxclude)
            if tmpwrong1 or tmpwrong2 or tmpwrong3 or tmpwrong4:
                aicc[i] = 99 # overwrite the AICc value

            # check if they are the same sign
            tmpsign = ((tmpr_clim.statistic > 0) == (tmpr_climDT.statistic > 0))
            tmppvalue = (tmpr_climDT.pvalue < 0.1) # check significance
            #if tmpsign==False:
            #    print('\tSIGN CHANGE ' + label + '\t'+
            #              'r={:.5f} \t r={:.5f}'.format(tmpr_clim.statistic, tmpr_climDT.statistic))
            #if tmppvalue==False:
            #    print('\tDT INSIGNIFICANT ' + label + '\t'+
            #              'p={:.5f} \t p={:.5f}'.format(tmpr_clim.pvalue, tmpr_climDT.pvalue))
            if (tmpsign==False) or (tmppvalue==False):
                aicc[i] = 99 # overwrite the AICc value
                #print('\tOverwrite AICc')

        # get minimum
        [iminaicc, minaicc] = np.nanargmin(aicc), np.nanmin(aicc)
        # check aicc value to previous value
        diffaicc = minaicc - tmpaicc[-1]
        # check if diff aicc is small enough before continuing
        #print('diffaicc = {:} - {:}'.format(minaicc, tmpaicc[-1]))
        if (diffaicc >= -2):
            #print('difference is too big, EXIT ' + str(diffaicc)) 
            continue # exit loop
        else:
            # continue this part if aicc < -2
            tmpi.append(iminaicc) # add min to list of variables
            tmpaicc.append(minaicc) # add aicc value to list
            f.write(labels[iminaicc] + '\n')
            print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
                      str(np.round(tmpr[iminaicc], decimals=3)))

            # calculate residuals from this current model
            tmpX = dfframesfor[dfnames[iregion]].iloc[:,tmpi] # all the variables
            f_vars.write(labels[iminaicc] + '\n' + str(tmpX.iloc[:,-1:].values.flatten()) + '\n') # write var data to file
            reg = LinearRegression()
            reg.fit(tmpX, tmpfire) # fit the final model
            tmppredictprior = reg.predict(tmpX)

    # store the final model
    tmpX = dfframesfor[dfnames[iregion]].iloc[:,tmpi] 
    reg.fit(tmpX, tmpfire) # fit the final model
    filename = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.sav' # save model
    joblib.dump(reg, filename) # save model        

    tmploadmodel = joblib.load(filename)
    tmpcoef = tmploadmodel.coef_
    tmpintercept = tmploadmodel.intercept_
    tmppred = tmploadmodel.predict(tmpX)
    print('R-value ' + str(pearsonr(tmpfire.flatten(), tmppred.flatten())[0]))
    # write to file
    f_vars.write('Model parameters \n' + str(tmpcoef) + '\n')
    f_vars.write('Model intercept\n' + str(tmpintercept) + '\n')
    f_vars.write('Burned area \n' + str(tmpfire) + '\n')
    f_vars.write('Model Burned area prediction \n' + str(tmppred))
    dfmodelfor[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list0.
    f_vars.close() # close variable data file
f.close()

FOREST
+++ model for allwestUS	 AICC 	 r-value
vpd y0 mo 7-9	-41.406	0.831
vpd y0 mo 4-6	-59.26	0.903


tmax y0 mo 10-12	-68.641	0.93


R-value 0.9295324531815231
+++ model for ecoprov1	 AICC 	 r-value
	AICc POSITIVE--EXIT
prec y-1 mo 1-3	99.0	0.053
R-value nan
+++ model for ecoprov2	 AICC 	 r-value
vpd y0 mo 4-6	-11.19	0.573


tmax y0 mo 7-9	-21.804	0.72


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


...

R-value 0.8128179430950827
+++ model for ecoprov18	 AICC 	 r-value
	AICc POSITIVE--EXIT
tmax y0 mo 10-12	-0.847	0.353
R-value nan
+++ model for ecoprov19	 AICC 	 r-value
vpd y0 mo 7-9	-18.849	0.669
tmax y0 mo 4-6	-36.061	0.816
prec y-1 mo 1-3	-42.44	0.857


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


vpd y0 mo 10-12	-50.229	0.892
R-value 0.8923381899036297
+++ model for ecoprov20	 AICC 	 r-value
vpd y0 mo 7-9	-36.888	0.808
solar y0 mo 1-3	-47.241	0.866
wind y0 mo 1-3	-49.846	0.883


R-value 0.8830834013248292
+++ model for ecoprov21	 AICC 	 r-value
	AICc POSITIVE--EXIT
tmax y0 mo 1-3	-2.774	0.408
R-value nan


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [15]:
# same model for NONFOREST area
print('NONFOREST')
landname = 'non'
modeloutputfile = model_string + 'modeloutput_burnarea_'+landname+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesnon['allwestUS'].columns.values # each climate variable

for iregion in np.arange(len(province_num)):
    print('+++ model for ' + dfnames[iregion] + '\t AICC \t r-value')
    f.write('+++'+dfnames[iregion]+'\n')
    varoutputfile = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.txt' # save variables in model
    f_vars = open(varoutputfile, 'w') # print to variable data file
    tmpr = np.zeros(len(labels)) # storage for regression correlation
    tmpr_clim = np.zeros(len(labels)) # storage for r corr, clim vs. BA
    tmpr_climDT = np.zeros(len(labels)) # storage for r corr, climDT vs. BA_DT
    aicc = np.zeros(len(labels)) # storage for aicc values for each set of reg fit
    tmpaicc = [] # storage for aicc values for each model
    tmpi = [] # empty list of indices for variables

    # get observed ecoregion burned area, but set min value
    tmpfire = np.asarray(np.ma.masked_invalid(np.log10(wuminon.iloc[:,iregion])).filled(0))
    tmpfiresum = wuminon.iloc[:,iregion].sum()
    tmpfiremin = tmpfire[tmpfire!=0].min()
    tmpfire[tmpfire==0] = tmpfiremin # replace 0 with minimum nonzero value
    tmpfireDT = annualDetrend(tmpfire)
    
    for i, label in enumerate(labels):
        tmpclim = np.asarray(dfframesnon[dfnames[iregion]][label].values.reshape(-1,1))
        tmpclimDT = annualDetrend(tmpclim.flatten())
        reg = LinearRegression() # least squares
        reg.fit(tmpclim, tmpfire)
        tmppredict = reg.predict(tmpclim)
        tmpr[i] = pearsonr(tmpfire.flatten(), tmppredict.flatten()).statistic
        # calculate AICc for each
        nyears = len(tmpfire)
        npredictors = reg.n_features_in_
        aicc[i] = getAICc(tmpr[i], nyears, npredictors)
        # compare signs
        tmpr_clim = pearsonr(tmpclim.flatten(), tmpfire)
        tmpr_climDT = pearsonr(tmpclimDT, tmpfireDT)
        # check if the correlation is right sign physically
        tmplabeljustvar = label.split(' ')[0]
        tmplabeljustyr = label.split(' ')[1]
        tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
        # wrong signs
        tmpwrong3 = (tmpr_clim.statistic < 0) & (tmplabeljustvar in clim_poscorrboth)
        tmpwrong4 = (tmpnameyrlabel in clim_correxclude)
        if tmpwrong3 or tmpwrong4:
            aicc[i] = 99 # overwrite the AICc value
            
        # check if they are the same sign
        tmpsign = ((tmpr_clim.statistic > 0) == (tmpr_climDT.statistic > 0))
        tmppvalue = (tmpr_climDT.pvalue < 0.1) # check significance
        #if tmpsign==False:
        #    print('\tSIGN CHANGE ' + label + '\t'+
        #              'r={:.5f} \t r={:.5f}'.format(tmpr_clim.statistic, tmpr_climDT.statistic))
        #if tmppvalue==False:
        #    print('\tDT INSIGNIFICANT ' + label + '\t'+
        #              'p={:.5f} \t p={:.5f}'.format(tmpr_clim.pvalue, tmpr_climDT.pvalue))
        if (tmpsign==False) or (tmppvalue==False):
            aicc[i] = 99 # overwrite the AICc value
            #print('\tOverwrite AICc')
    # get minimum
    [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
    tmpi.append(iminaicc) # add min to list of variables
    tmpaicc.append(minaicc) # add aicc value to list
    f.write(labels[iminaicc] + '\n')

    if (minaicc>0) or (tmpfiresum<minfiresum): # we overwrite the model and exit
        tmpclim = np.asarray(dfframesnon[dfnames[iregion]].iloc[:,tmpi].values.reshape(-1,1))
        reg = LinearRegression()
        reg.fit(tmpclim, tmpfire)
        reg.coef_ = np.asarray([0]) # change coeff to 0
        reg.intercept_ = np.asarray(tmpfire.mean())
        f_vars.write(labels[iminaicc] + '\n' + str(tmpclim.flatten()) + '\n') # write variable data to file
        print('\tAICc POSITIVE--EXIT')
        print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
              str(np.round(tmpr[iminaicc], decimals=3)))
        
        # store the final model
        filename = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.sav' # save model
        joblib.dump(reg, filename) # save model        

        tmpcoef = reg.coef_
        tmpintercept = reg.intercept_
        tmppred = reg.predict(tmpclim)
        print('R-value ' + str(pearsonr(tmpfire.flatten(), tmppred.flatten())[0]))
        # write to file
        f_vars.write('Model parameters \n' + str(tmpcoef) + '\n')
        f_vars.write('Model intercept\n' + str(tmpintercept) + '\n')
        f_vars.write('Burned area \n' + str(tmpfire) + '\n')
        f_vars.write('Model Burned area prediction \n' + str(tmppred))
        dfmodelnon[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list0.
        f_vars.close() # close variable data file
        continue # leave this loop
    
    # calculate residuals from one-variable model
    tmpclim = dfframesnon[dfnames[iregion]].iloc[:,tmpi]
    f_vars.write(labels[iminaicc] + '\n' + str(tmpclim.values.flatten()) + '\n') # write variable data to file
    reg = LinearRegression()
    reg.fit(tmpclim, tmpfire)
    tmppredictprior = reg.predict(tmpclim)

    print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
              str(np.round(tmpr[iminaicc], decimals=3)))
    
    # define starting difference in aicc to get the for-loop to start
    diffaicc = -3
    # multiple regression using remaining predictor, until aicc is not improved
    while diffaicc < -2:
        # add another variable to get new model
        # calculate residuals
        tmpresids = np.log10((10**tmpfire)/(10**tmppredictprior))
        # go through each variable and get the correlation
        tmpr = np.zeros(len(labels)) # storage for regression correlation
        for i, label in enumerate(labels):
            tmpclimnew = np.asarray(dfframesnon[dfnames[iregion]][label]).reshape(-1,1)
            tmpclimprior = dfframesnon[dfnames[iregion]].iloc[:,tmpi]
            tmpX = np.hstack((tmpclimprior, tmpclimnew)) # all X variables
            tmpclimnewDT = annualDetrend(tmpclimnew.flatten())
            tmpresidsDT = annualDetrend(tmpresids)
            reg = LinearRegression() # least squares
            reg.fit(tmpX, tmpfire)
            tmppredict = reg.predict(tmpX)
            tmpr[i] = pearsonr(tmpfire.flatten(), tmppredict.flatten()).statistic
            # calculate AICc for each
            nyears = len(tmpfire)
            npredictors = reg.n_features_in_
            aicc[i] = getAICc(tmpr[i], nyears, npredictors)
            # compare signs
            tmpr_clim = pearsonr(tmpclimnew.flatten(), tmpresids)
            tmpr_climDT = pearsonr(tmpclimnewDT, tmpresidsDT)
            # check if correlation is the right sign physically
            tmplabeljustvar = label.split(' ')[0]
            tmplabeljustyr = label.split(' ')[1]
            tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
            # wrong signs
            tmpwrong3 = (tmpr_clim.statistic < 0) & (tmplabeljustvar in clim_poscorrboth)
            tmpwrong4 = (tmpnameyrlabel in clim_correxclude)
            if tmpwrong3 or tmpwrong4:
                aicc[i] = 99 # overwrite the AICc value
            # check if they are the same sign
            tmpsign = ((tmpr_clim.statistic > 0) == (tmpr_climDT.statistic > 0))
            tmppvalue = (tmpr_climDT.pvalue < 0.1) # check significance
            #if tmpsign==False:
            #    print('\tSIGN CHANGE ' + label + '\t'+
            #              'r={:.5f} \t r={:.5f}'.format(tmpr_clim.statistic, tmpr_climDT.statistic))
            #if tmppvalue==False:
            #    print('\tDT INSIGNIFICANT ' + label + '\t'+
            #              'p={:.5f} \t p={:.5f}'.format(tmpr_clim.pvalue, tmpr_climDT.pvalue))
            if (tmpsign==False) or (tmppvalue==False):
                aicc[i] = 99 # overwrite the AICc value
                #print('\tOverwrite AICc')

        # get minimum
        [iminaicc, minaicc] = np.nanargmin(aicc), np.nanmin(aicc)
        # check aicc value to previous value
        diffaicc = minaicc - tmpaicc[-1]
        # check if diff aicc is small enough before continuing
        #print('diffaicc = {:} - {:}'.format(minaicc, tmpaicc[-1]))
        if (diffaicc >= -2):
            #print('difference is too big, EXIT ' + str(diffaicc)) 
            continue # exit loop
        else:
            # continue this part if aicc < -2
            tmpi.append(iminaicc) # add min to list of variables
            tmpaicc.append(minaicc) # add aicc value to list
            f.write(labels[iminaicc] + '\n')
            print(labels[iminaicc]+ '\t' + str(np.round(minaicc, decimals=3))+ '\t' + 
                      str(np.round(tmpr[iminaicc], decimals=3)))

            # calculate residuals from this current model
            tmpX = dfframesnon[dfnames[iregion]].iloc[:,tmpi] # all the variables
            f_vars.write(labels[iminaicc] + '\n' + str(tmpX.iloc[:,-1:].values.flatten()) + '\n') # write var data to file
            reg = LinearRegression()
            reg.fit(tmpX, tmpfire) # fit the final model
            tmppredictprior = reg.predict(tmpX)

    # store the final model
    tmpX = dfframesnon[dfnames[iregion]].iloc[:,tmpi] 
    reg.fit(tmpX, tmpfire) # fit the final model
    filename = model_string + 'model_'+landname+'_burnedarea_'+dfnames[iregion]+'.sav' # save model
    joblib.dump(reg, filename) # save model        

    tmploadmodel = joblib.load(filename)
    tmpcoef = tmploadmodel.coef_
    tmpintercept = tmploadmodel.intercept_
    tmppred = tmploadmodel.predict(tmpX)
    print('R-value ' + str(pearsonr(tmpfire.flatten(), tmppred.flatten())[0]))
    # write to file
    f_vars.write('Model parameters \n' + str(tmpcoef) + '\n')
    f_vars.write('Model intercept\n' + str(tmpintercept) + '\n')
    f_vars.write('Burned area \n' + str(tmpfire) + '\n')
    f_vars.write('Model Burned area prediction \n' + str(tmppred))
    dfmodelnon[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list0.
    f_vars.close() # close variable data file
f.close()

NONFOREST
+++ model for allwestUS	 AICC 	 r-value
vpd y0 mo 7-9	-7.338	0.509
prec y-1 mo 4-6	-15.877	0.663


R-value 0.6627842609922346
+++ model for ecoprov1	 AICC 	 r-value
prec y-1 mo 10-12	-7.226	0.506
prec y-1 mo 1-3	-15.185	0.655
tmax y0 mo 7-9	-17.443	0.703


R-value 0.7033310068250582
+++ model for ecoprov2	 AICC 	 r-value
wind y0 mo 1-3	-3.209	0.419
tmax y0 mo 7-9	-5.83	0.524
wetdays y0 mo 7-9	-10.893	0.634


prec y-1 mo 4-6	-14.583	0.701
R-value 0.7013135257984965
+++ model for ecoprov3	 AICC 	 r-value
	AICc POSITIVE--EXIT
prec y-1 mo 1-3	99.0	0.212
R-value nan
+++ model for ecoprov4	 AICC 	 r-value
vpd y0 mo 10-12	-5.113	0.464


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


...

/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


vpd y0 mo 4-6	-12.04	0.618
vpd y0 mo 10-12	-14.624	0.676
R-value 0.6758656002000364
+++ model for ecoprov20	 AICC 	 r-value
vpd y0 mo 4-6	-21.598	0.697
solar y0 mo 7-9	-29.579	0.778


R-value 0.7780726015276273
+++ model for ecoprov21	 AICC 	 r-value
prec y-1 mo 7-9	-6.278	0.488
wind y0 mo 1-3	-11.49	0.61
prec y-1 mo 1-3	-13.805	0.667
solar y0 mo 10-12	-16.347	0.717


prec y-1 mo 4-6	-18.594	0.757
R-value 0.7571957086875725


/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [16]:
# print last time this model was updated
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print('Outputting for folder = ', climindname)
print("Model last run =", dt_string)

Outputting for folder =  patch125-155_nino3-34
Model last run = 18/11/2025 10:52:33
